<a href="https://colab.research.google.com/github/mfernandaazambuja12-cyber/CEP_2026/blob/main/MVP_CEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Previsão de Falha de Máquina com CEP e Machine Learning
## Dataset: AI4I 2020 Predictive Maintenance (Kaggle)
**Autor:** Maria Fernanda Oliveira Azambuja

**Data:** 22 de Abril de 2026  

**Tipo:** Classificação Binária  


### Objetivo
Desenvolver um modelo preditivo para detectar falhas em máquinas industriais,
integrando Controle Estatístico de Processos (CEP) com Machine Learning.


In [ ]:
# ========================================
# ETAPA 0 — INSTALAÇÃO E CONFIGURAÇÃO
# ========================================

# Instalar bibliotecas (se necessário no Colab)
!pip install xgboost imbalanced-learn -q

# Imports principais
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# Reprodutibilidade — SEEDS FIXOS
import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print('Ambiente configurado com sucesso!')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')


Ambiente configurado com sucesso!
NumPy: 2.0.2 | Pandas: 2.2.2


## Etapa 1: Dataset e Fontes

**Fonte:** Kaggle — AI4I 2020 Predictive Maintenance Dataset

**URL:** https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020


O dataset contém 10.000 registros de sensores de máquinas industriais (tipo CNC),
com 6 features de processo e variável-alvo binária indicando falha (1) ou não (0).

**Variáveis:**
- Air temperature [K]: Temperatura do ar
- Process temperature [K]: Temperatura do processo
- Rotational speed [rpm]: Velocidade de rotação
- Torque [Nm]: Torque aplicado
- Tool wear [min]: Desgaste da ferramenta
- Type: Qualidade do produto (L=Low, M=Medium, H=High)
- Machine failure: Target (0 = OK, 1 = Falha)

In [ ]:
# ========================================
# ETAPA 1 — CARREGAMENTO DOS DADOS
# ========================================

import kagglehub

# Download latest version
path = kagglehub.dataset_download("stephanmatzka/predictive-maintenance-dataset-ai4i-2020")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'predictive-maintenance-dataset-ai4i-2020' dataset.
Path to dataset files: /kaggle/input/predictive-maintenance-dataset-ai4i-2020


In [ ]:
# Inspeção inicial

df = kagglehub.dataset_download("stephanmatzka/predictive-maintenance-dataset-ai4i-2020")

# Inspeção inicial
print('=== SHAPE ===')
print(df.shape)

print('\n=== TIPOS ===')
print(df.dtypes)

print('\n=== VALORES AUSENTES ===')
print(df.isnull().sum())

print('\n=== ESTATÍSTICAS DESCRITIVAS ===')
df.describe()

In [ ]:
# Distribuição do target
print('=== DISTRIBUIÇÃO DE MACHINE FAILURE ===')
print(df['Machine failure'].value_counts())
print(f"\nProporção de falhas: {df['Machine failure'].mean()*100:.2f}%")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
df['Machine failure'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#2196F3','#F44336'], edgecolor='white')
axes[0].set_title('Distribuição: Falha vs Sem Falha', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['Sem Falha (0)', 'Falha (1)'], rotation=0)
axes[0].set_ylabel('Quantidade')

# Pizza
df['Machine failure'].value_counts().plot(kind='pie', ax=axes[1],
    autopct='%1.1f%%', colors=['#2196F3','#F44336'],
    labels=['Sem Falha', 'Falha'])
axes[1].set_title('Proporção de Classes', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('distribuicao_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observação: Dataset fortemente desbalanceado — estratégia de balanceamento necessária!')

## Etapa 2: Análise Exploratória de Dados (EDA)

Objetivo: compreender a distribuição das variáveis, identificar padrões,
correlações e possíveis problemas nos dados antes da modelagem.

In [ ]:
# Distribuição das features numéricas
numeric_cols = ['Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=40, color='#2196F3', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frequência')
    # Adicionar média
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Média: {df[col].mean():.1f}')
    axes[i].legend(fontsize=9)

# Distribuição do tipo de produto
df['Type'].value_counts().plot(kind='bar', ax=axes[5], color=['#FF9800','#4CAF50','#9C27B0'])
axes[5].set_title('Tipo de Produto (L/M/H)', fontsize=11, fontweight='bold')
axes[5].set_xlabel('Tipo')
axes[5].set_xticklabels(['L (Low)','M (Medium)','H (High)'], rotation=0)

plt.suptitle('Distribuição das Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distribuicao_features.png', dpi=150, bbox_inches='tight')
plt.show()